In [ ]:
%%bash
set -euo pipefail

# Cell 1: Setup venv + clone repo + install base stack
REPO_URL="${REPO_URL:-https://github.com/AneKazek/malesbgt.git}"
REPO_BRANCH="${REPO_BRANCH:-main}"

apt-get update -qq
apt-get install -y python3.11 python3.11-venv python3.11-distutils git gcc g++ build-essential python3.11-dev > /dev/null

mkdir -p /kaggle/temp
cd /kaggle/temp

python3.11 -m venv .venv
source .venv/bin/activate
python -m pip install --upgrade pip setuptools wheel

if [ -d kcv-tts ]; then rm -rf kcv-tts; fi
git clone --depth 1 --branch "$REPO_BRANCH" "$REPO_URL" kcv-tts

cd kcv-tts
test -f requirements-py311-torch260-cu124.txt
pip install -r requirements-py311-torch260-cu124.txt
pip install -e . --no-deps

python - <<'PY'
import torch
print('torch version:', torch.__version__)
print('torch cuda:', torch.version.cuda)
print('torch cxx11abi:', torch._C._GLIBCXX_USE_CXX11_ABI)
PY

In [ ]:
%%bash
set -euo pipefail

# Cell 2: Build wheels from source, install local wheels, export archive
VENV=/kaggle/temp/.venv
WHEEL_DIR=/kaggle/working/wheelhouse
ARCHIVE=/kaggle/working/wheelhouse_mamba_torch260_cu124.tar.gz

source "$VENV/bin/activate"
pip uninstall -y causal-conv1d mamba-ssm 2>/dev/null || true
pip cache purge || true

python - <<'PY'
import site, glob, os, shutil
for d in site.getsitepackages():
    for pattern in ["causal_conv1d*", "mamba_ssm*", "causal_conv1d_cuda*", "selective_scan_cuda*"]:
        for p in glob.glob(os.path.join(d, pattern)):
            try:
                if os.path.isdir(p):
                    shutil.rmtree(p, ignore_errors=True)
                else:
                    os.remove(p)
            except Exception:
                pass
PY

mkdir -p "$WHEEL_DIR"
export CXXFLAGS="-D_GLIBCXX_USE_CXX11_ABI=0"
export CFLAGS="-D_GLIBCXX_USE_CXX11_ABI=0"
export LDFLAGS="-D_GLIBCXX_USE_CXX11_ABI=0"
export LD_LIBRARY_PATH="/usr/local/cuda/lib64:/usr/local/cuda-12.4/lib64:${LD_LIBRARY_PATH:-}"

pip wheel --no-deps --no-cache-dir --no-build-isolation --no-binary=:all: --wheel-dir "$WHEEL_DIR" causal-conv1d==1.6.1
pip wheel --no-deps --no-cache-dir --no-build-isolation --no-binary=:all: --wheel-dir "$WHEEL_DIR" mamba-ssm==2.3.1

pip install --no-deps --no-cache-dir --force-reinstall "$WHEEL_DIR"/causal_conv1d-*.whl "$WHEEL_DIR"/mamba_ssm-*.whl

python - <<'PY'
import torch, causal_conv1d, mamba_ssm
print('OK torch', torch.__version__, 'cuda', torch.version.cuda, 'abi', torch._C._GLIBCXX_USE_CXX11_ABI)
print('OK causal_conv1d', getattr(causal_conv1d, '__version__', 'n/a'))
print('OK mamba_ssm', getattr(mamba_ssm, '__version__', 'n/a'))
PY

tar -czf "$ARCHIVE" -C /kaggle/working wheelhouse
ls -lh "$WHEEL_DIR"
ls -lh "$ARCHIVE"

In [ ]:
%%bash
set -euo pipefail

# Cell 3: Reuse on next run (no compile)
# Attach your uploaded wheel dataset and set WHEEL_SRC_DIR path below
VENV=/kaggle/temp/.venv
WHEEL_SRC_DIR="${WHEEL_SRC_DIR:-/kaggle/input/your-wheelhouse-dataset/wheelhouse}"

if [ ! -d "$WHEEL_SRC_DIR" ]; then
  echo "Wheel source directory not found: $WHEEL_SRC_DIR"
  echo "Set WHEEL_SRC_DIR to your mounted dataset path."
  exit 0
fi

source "$VENV/bin/activate"
pip install --no-deps --no-cache-dir --force-reinstall "$WHEEL_SRC_DIR"/causal_conv1d-*.whl "$WHEEL_SRC_DIR"/mamba_ssm-*.whl

python - <<'PY'
import torch, causal_conv1d, mamba_ssm
print('OK torch', torch.__version__, 'cuda', torch.version.cuda, 'abi', torch._C._GLIBCXX_USE_CXX11_ABI)
print('OK causal_conv1d', getattr(causal_conv1d, '__version__', 'n/a'))
print('OK mamba_ssm', getattr(mamba_ssm, '__version__', 'n/a'))
PY